In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.interpolate import interp1d
import imageio
import os

# Function to generate plot for a given dataset and time
def generate_plot(ax, df1, time, title):
    # Create a color map for the vertical strips based on mineral content
    norm = Normalize(vmin=70, vmax=99)  # Normalize between 65% and 99%
    cmap = plt.cm.gray  # Use a gray colormap (black at low values, white at high values)

    # Set the y-axis (length) range to 0-2 microns
    y_length = 1
    y_positions = np.linspace(0, y_length, 100)  # Create 100 equally spaced positions on the y-axis

    # Sort the DataFrame by position
    df1 = df1.sort_values(by='Position')

    # Interpolate positions for smooth plotting
    x_positions = df1['Position'].values
    mineral_content = (1 - df1['Porosity']).values * 100

    # Interpolate over the x positions
    interpolator = interp1d(x_positions, mineral_content, kind='linear', fill_value='extrapolate')
    x_new = np.linspace(x_positions.min(), x_positions.max(), 3000)
    y_new = interpolator(x_new)

    # Plot smooth vertical strips
    for idx in range(len(x_new) - 1):
        x_start = x_new[idx]
        x_end = x_new[idx + 1]
        mineral_percentage = y_new[idx]
        color = cmap(norm(mineral_percentage))
        ax.fill_betweenx(y_positions, x_start, x_end, color=color, linewidth=0, edgecolor='none')

    ax.set_ylabel('Length ($\mu m$)', fontsize=16)
    ax.set_xlabel('Depth of Enamel ($\mu m$)', fontsize=16)
    ax.set_xlim(0, 12)
    ax.set_ylim(0, y_length)
    ax.set_title(title, fontsize=14)

    # Create a color bar
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Mineral (%)', fontsize=16)


# Function to save plots for each time and generate a GIF
def generate_gif(datasets, output_gif="output.gif", duration=5.0):
    filenames = []
    
    # Loop over each time stamp
    for time in datasets[list(datasets.keys())[0]][1]:  # Use time stamps from the first dataset
        fig, axes = plt.subplots(len(datasets), figsize=(10, 4))  # Create a subplot for each dataset

        if len(datasets) == 1:
            axes = [axes]  # Ensure axes is a list even for a single subplot
        
        # Loop over each dataset and corresponding axis
        for ax, (dataset_name, (df, time_stamps)) in zip(axes, datasets.items()):
            filtered_data = df[df['Time in min'] == time]
            title = f"at t = {time} min"
            generate_plot(ax, filtered_data, time, title)

        # Save the figure as an image
        filename = f"frame_{time}.png"
        plt.tight_layout()
        plt.savefig(filename)
        filenames.append(filename)
        plt.close()

    # Create GIF from saved images
    with imageio.get_writer(output_gif, mode='I', duration=duration, loop=0) as writer:
        for filename in filenames:
            image = imageio.imread(filename)
            writer.append_data(image)

    # Remove the individual frame files after the GIF is created
    for filename in filenames:
        os.remove(filename)

# Main function
def main():
    # Read the CSV files
    df = pd.read_csv('Remin_demin_short_ca1p32_HighKhap.csv')
    # df1 = pd.read_csv('Remin_demin_100_250_100_15cycle_Dx_short_ca0p04_always.csv')

    # Define the time stamps and datasets
    time_stamp_long = [10 * i for i in range(0, 46)]
    datasets = {
        'Control 2 min Glu Pulse': (df, time_stamp_long)
    }

    # Generate GIF
    generate_gif(datasets, output_gif="mineral_content_over_time_5pulse.gif", duration=5.0)

if __name__ == "__main__":
    main()


<>:40: SyntaxWarning: invalid escape sequence '\m'
<>:41: SyntaxWarning: invalid escape sequence '\m'
<>:40: SyntaxWarning: invalid escape sequence '\m'
<>:41: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_6056/3042267649.py:40: SyntaxWarning: invalid escape sequence '\m'
  ax.set_ylabel('Length ($\mu m$)', fontsize=16)
/tmp/ipykernel_6056/3042267649.py:41: SyntaxWarning: invalid escape sequence '\m'
  ax.set_xlabel('Depth of Enamel ($\mu m$)', fontsize=16)
/tmp/ipykernel_6056/3042267649.py:80: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(filename)
